# Phase 3 CoT Reasoning Fine-Tuning — Qwen2.5-Coder-7B SFT-merged (Kaggle & Colab)

Self-contained notebook for **Phase 3 Chain-of-Thought (CoT) reasoning fine-tuning** on `Aniket200325/coder-reasoning-cot-v1` dataset (~25K samples).
Optimized for **Kaggle GPUs** (2x T4 / P100 / L4) and **Google Colab**. Automatically syncs intermediate checkpoints to **Hugging Face Hub** for seamless resume across Kaggle session restarts.

### Lineage & Base Model
- **Base model**: `Aniket200325/coder-qwen25-coder-7b-sft-merged` (Phase 2 SFT-merged BF16)
- **Dataset**: `Aniket200325/coder-reasoning-cot-v1` (train + heldout eval split `eval_heldout`)
- **Hub Adapter ID**: `Aniket200325/coder-qwen25-coder-7b-cot-qlora-v1`


### Installation


In [1]:
%%capture
import os, re

# BEFORE any Unsloth import — avoid fast CDN hang + stats probes
os.environ["UNSLOTH_STABLE_DOWNLOADS"] = "1"
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
for k in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
    os.environ.pop(k, None)
os.environ["PYTHONUNBUFFERED"] = "1"

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {
        "2.10": "0.0.34", "2.9": "0.0.33.post1", "2.8": "0.0.32.post2", "2.11": "0.0.35",
    }.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps transformers
    !pip install --no-deps --upgrade "torchao>=0.16.0"

# Install FlashAttention-2 for A100 (~1.5h/epoch speed at 8192 seq len)
!pip install flash-attn --no-build-isolation


In [2]:
import os
import torch, transformers
print("transformers", transformers.__version__, "torch", torch.__version__)
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"GPUs detected: {num_gpus}")
if num_gpus > 0:
    for i in range(num_gpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("  GPU: CPU")
print("Download env OK:", {
    k: os.environ.get(k) for k in (
        "UNSLOTH_STABLE_DOWNLOADS", "UNSLOTH_DISABLE_STATISTICS",
        "HF_HUB_DISABLE_XET", "HF_HUB_ENABLE_HF_TRANSFER",
    )
})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:298: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


transformers 5.13.1 torch 2.11.0+cu128
GPUs detected: 1
  GPU 0: NVIDIA A100-SXM4-40GB
Download env OK: {'UNSLOTH_STABLE_DOWNLOADS': '1', 'UNSLOTH_DISABLE_STATISTICS': '1', 'HF_HUB_DISABLE_XET': '1', 'HF_HUB_ENABLE_HF_TRANSFER': '1'}


### Config + auth

Fill `HF_TOKEN` (or Colab Secret). Set `SMOKE = True` for a short dry-run (masking + packing gates); `False` for a full SFT session with **1h Hugging Face checkpoint sync** + `RESUME=auto`. Keep `RUN_FINAL_MERGE = False` until the last training session.


In [ ]:
import os, json, time, shutil, math, gc, random
from pathlib import Path
from dataclasses import dataclass, asdict

# ── fill these ──────────────────────────────────────────────────────────────
HF_TOKEN = ""  # or Colab Secrets / userdata
USE_DRIVE = True  # Colab convenience only; Kaggle uses Hugging Face checkpoints
SMOKE = True  # True → short smoke; False → full SFT session
RUN_FINAL_MERGE = False  # Cell 22 gate; flip True only on last run
FORCE_MERGE = False  # Cell 22: overwrite existing merge out_dir

MODEL_HUB = "Aniket200325/coder-qwen25-coder-7b-sft-qlora-v1-merged"
MODEL_DRIVE = Path("/content/drive/MyDrive/coder-qwen25-coder-7b-sft-qlora-v1-merged")
DATASET = "Aniket200325/coder-reasoning-cot-v1"
HUB_ADAPTER_ID = "Aniket200325/coder-qwen25-coder-7b-cot-qlora-v1"
HUB_MERGED_SFT_ID = ""  # optional private Hub for Cell 22 merged BF16

# Auto-tune max_seq_length for hardware
# Tesla T4 (Turing) lacks FlashAttention-2 kernel tiling, making 8192 O(N^2) slow.
# 4096 seq len on T4 runs 4x faster and covers >96% of CoT traces.
IS_TURING_GPU = torch.cuda.is_available() and any("T4" in torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count()))
MAX_SEQ_LEN = 4096 if IS_TURING_GPU else 8192
PACKING = False
LOAD_IN_4BIT = True
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LR = 1.5e-5  # Optimal learning rate for effective batch 32
SEED = 42

BATCH = 2 if SMOKE else 4  # 2 for fast 6-min smoke test; 4 for full run
ACCUM = 4 if SMOKE else 8  # Effective batch 8 for smoke; 32 for full run
MAX_STEPS = 30 if SMOKE else None  # None → num_train_epochs
NUM_TRAIN_EPOCHS = 2  # 2 epochs over ~25k reasoning traces (~1,560 total steps at eff. batch 32)
EVAL_STEPS = 10 if SMOKE else 200
SAVE_STEPS = 15 if SMOKE else 500
LOGGING_STEPS = 1 if SMOKE else 10
CKPT_MINUTES = 5.0 if SMOKE else 60.0  # 1h timed saves (Phase 1 pattern)
WALL_CLOCK_HOURS = 11.5  # stop before Colab ~12h kill; 0 = disable
RESUME = "none" if SMOKE else "auto"

OUT_DIR = Path(
    "/content/ckpts/qwen25-coder-7b-phase3-cot-smoke"
    if SMOKE
    else "/content/ckpts/qwen25-coder-7b-phase3-cot"
)
MERGE_OUT_DIR = Path("/content/drive/MyDrive/coder-qwen25-coder-7b-cot-merged")

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").is_dir()
IS_COLAB = "COLAB_" in "".join(os.environ.keys()) or Path("/content").is_dir()
USE_DRIVE = bool(USE_DRIVE and IS_COLAB and not IS_KAGGLE)
if IS_KAGGLE:
    OUT_DIR = Path("/kaggle/working/ckpts/qwen25-coder-7b-phase3-cot-smoke" if SMOKE else "/kaggle/working/ckpts/qwen25-coder-7b-phase3-cot")
    MERGE_OUT_DIR = Path("/kaggle/working/coder-qwen25-coder-7b-cot-merged")
# ────────────────────────────────────────────────────────────────────────────

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception:
        HF_TOKEN = ""
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or ""

assert HF_TOKEN, (
    "Set HF_TOKEN (Hugging Face token) — private Hub model + dataset require auth"
)
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

DRIVE_CKPT = None
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_CKPT = Path(
            "/content/drive/MyDrive/coder-qwen25-coder-7b-phase3-cot-smoke"
            if SMOKE
            else "/content/drive/MyDrive/coder-qwen25-coder-7b-phase3-cot"
        )
        DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        print(f"Notice: Google Drive unavailable ({e}). Continuing without Drive mirroring.")
        USE_DRIVE = False
        DRIVE_CKPT = None

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Throughput / Ampere knobs
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("SMOKE:", SMOKE, "| OUT:", OUT_DIR, "| DRIVE:", DRIVE_CKPT)
print(
    "batch", BATCH, "accum", ACCUM, "effective", BATCH * ACCUM,
    "| seq", MAX_SEQ_LEN, "| packing", PACKING,
    "| CKPT_MINUTES", CKPT_MINUTES, "| resume", RESUME,
)
print("wall_clock_stop_h", WALL_CLOCK_HOURS, "| RUN_FINAL_MERGE", RUN_FINAL_MERGE)
print("HUB_ADAPTER_ID", HUB_ADAPTER_ID)



Mounted at /content/drive
SMOKE: True | OUT: /content/ckpts/qwen25-coder-7b-phase3-cot-smoke | DRIVE: /content/drive/MyDrive/coder-qwen25-coder-7b-phase3-cot-smoke
batch 2 accum 4 effective 8 | seq 8192 | packing False | CKPT_MINUTES 5.0 | resume none
wall_clock_stop_h 11.5 | RUN_FINAL_MERGE False
HUB_ADAPTER_ID Aniket200325/coder-qwen25-coder-7b-cot-qlora-v1


### Load Phase-2 SFT-merged base (QLoRA 4-bit) + ChatML template

Prefer Drive merge if complete; else Hub snapshot. Weights start from the **Phase-2 SFT-merged** artifact and keep the Qwen ChatML template.


In [5]:
# Resolve CPT-merged domain base → prefetch → QLoRA load → ChatML template.
import os
os.environ.setdefault("UNSLOTH_STABLE_DOWNLOADS", "1")
os.environ.setdefault("UNSLOTH_DISABLE_STATISTICS", "1")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
for k in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
    os.environ.pop(k, None)

from huggingface_hub import snapshot_download
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

def local_model_complete(path: Path) -> bool:
    if not path.is_dir() or not (path / "config.json").is_file():
        return False
    for pattern in ("*.safetensors", "pytorch_model*.bin", "model*.bin"):
        for p in path.glob(pattern):
            if p.is_file() and p.stat().st_size > 1_000_000:
                return True
    return False

def is_adapter_only_dir(path: Path) -> bool:
    """True if dir looks like PEFT adapters without a full base config+weights."""
    if not path.is_dir():
        return False
    has_adapter = (path / "adapter_config.json").is_file()
    has_full = local_model_complete(path)
    return has_adapter and not has_full

MODEL_CACHE = Path("/content/models") / MODEL_HUB.replace("/", "--")

# 1) Resolve model path
resolved = None
if USE_DRIVE and local_model_complete(MODEL_DRIVE):
    resolved = MODEL_DRIVE
    print("Using Drive CPT-merged base:", resolved)
elif local_model_complete(MODEL_CACHE):
    resolved = MODEL_CACHE
    print("Using cached Hub snapshot:", resolved)
else:
    if USE_DRIVE and MODEL_DRIVE.exists() and is_adapter_only_dir(MODEL_DRIVE):
        raise SystemExit(
            f"MODEL_DRIVE looks like adapters only (adapter_config.json, no full weights): {MODEL_DRIVE}\n"
            "Point at the CPT-merged BF16 base (run fine-tune/merge_cpt_lora_colab.py first), "
            "not Phase-1 final/ LoRA."
        )
    if MODEL_CACHE.exists():
        print("Incomplete local dir; removing", MODEL_CACHE)
        shutil.rmtree(MODEL_CACHE, ignore_errors=True)
    MODEL_CACHE.parent.mkdir(parents=True, exist_ok=True)
    print("Prefetching", MODEL_HUB, "→", MODEL_CACHE)
    try:
        snapshot_download(
            repo_id=MODEL_HUB,
            local_dir=str(MODEL_CACHE),
            token=HF_TOKEN,
            resume_download=True,
            max_workers=8,
        )
    except Exception as e:
        raise SystemExit(
            f"Could not load CPT-merged base from Drive or Hub.\n"
            f"Drive complete={local_model_complete(MODEL_DRIVE) if USE_DRIVE else False}, "
            f"Hub prefetch error: {e}\n"
            "Run fine-tune/merge_cpt_lora_colab.py first, or ensure Hub "
            f"{MODEL_HUB} is reachable with your token."
        ) from e
    if not local_model_complete(MODEL_CACHE):
        raise SystemExit(
            f"Prefetch incomplete: {MODEL_CACHE}. "
            "Merge not done — see fine-tune/merge_cpt_lora_colab.py."
        )
    gb = sum(p.stat().st_size for p in MODEL_CACHE.rglob("*") if p.is_file()) / 1e9
    print(f"Prefetch complete ({gb:.1f} GB)")
    resolved = MODEL_CACHE

assert resolved is not None and local_model_complete(resolved), (
    "No complete CPT-merged base. Run fine-tune/merge_cpt_lora_colab.py."
)

# 2) Hard lineage checks
if is_adapter_only_dir(resolved):
    raise SystemExit(
        f"Resolved path is adapter-only (CPT LoRA?), not a merged base: {resolved}"
    )
if "instruct" in str(resolved).lower() or "Instruct" in MODEL_HUB:
    print(
        "WARNING: path/id contains 'Instruct' — Phase 3 should init from your "
        "Phase-2 SFT-merged weights, not the original upstream Instruct model.",
        flush=True,
    )

MODEL_DIR = Path(resolved)
print("MODEL_DIR:", MODEL_DIR)

# 3) Load QLoRA (Multi-GPU compatible)
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
load_kwargs = dict(
    model_name=str(MODEL_DIR),
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,  # auto bf16/fp16 compute
    load_in_4bit=LOAD_IN_4BIT,
    token=HF_TOKEN,
)
if num_gpus > 1 and "LOCAL_RANK" not in os.environ:
    load_kwargs["device_map"] = "auto"
load_kwargs["max_memory"] = {i: "13GB" for i in range(num_gpus)}

try:
    model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
except TypeError:
    load_kwargs.pop("device_map", None)
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
    except TypeError:
        load_kwargs.pop("token", None)
        try:
            model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
        except TypeError:
            load_kwargs.pop("max_seq_length", None)
            model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)

# 4) Chat template (weights stay merged Base; template only)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
assert getattr(tokenizer, "chat_template", None), (
    "tokenizer.chat_template is empty after get_chat_template"
)
_dummy = [
    {"role": "system", "content": "You are a helpful coding assistant."},
    {"role": "user", "content": "Say hi."},
    {"role": "assistant", "content": "Hello."},
]
_rendered = tokenizer.apply_chat_template(
    _dummy, tokenize=False, add_generation_prompt=False
)
assert "<|im_start|>assistant" in _rendered, (
    f"ChatML sanity failed — missing <|im_start|>assistant in:\n{_rendered[:400]}"
)
print("ChatML template OK; sample head:", repr(_rendered[:160]))

cfg = model.config
archs = list(getattr(cfg, "architectures", None) or [])
has_vision = hasattr(cfg, "vision_config")
print(
    "Loaded CPT-merged | tokenizer", type(tokenizer).__name__,
    f"| load_in_4bit={LOAD_IN_4BIT}",
)
print("architectures:", archs, "| vision_config:", has_vision)
assert not has_vision, "Model has vision_config — unexpected for this Phase 3 path."
assert not any(a.endswith("ForConditionalGeneration") for a in archs), (
    f"ConditionalGeneration architecture {archs} → wrong checkpoint."
)


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Prefetching Aniket200325/coder-qwen25-coder-7b-sft-qlora-v1-merged → /content/models/Aniket200325--coder-qwen25-coder-7b-sft-qlora-v1-merged


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Prefetch complete (15.2 GB)
MODEL_DIR: /content/models/Aniket200325--coder-qwen25-coder-7b-sft-qlora-v1-merged
==((====))==  Unsloth 2026.7.5: Fast Qwen2 patching. Transformers: 5.13.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

ChatML template OK; sample head: '<|im_start|>system\nYou are a helpful coding assistant.<|im_end|>\n<|im_start|>user\nSay hi.<|im_end|>\n<|im_start|>assistant\nHello.<|im_end|>\n'
Loaded CPT-merged | tokenizer Qwen2Tokenizer | load_in_4bit=True
architectures: ['Qwen2ForCausalLM'] | vision_config: False


### Fresh LoRA (do not load CPT adapters)

New PEFT scaffold only. On resume, Trainer reloads adapter + optimizer state from `checkpoint-*` after this cell.


In [ ]:
# Fresh LoRA — never PeftModel.from_pretrained(CPT adapters) here.
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
print("Fresh LoRA ready: r=", LORA_R, "alpha=", LORA_ALPHA, "| 4bit=", LOAD_IN_4BIT)
print("(Resume will overlay adapter/optimizer weights from checkpoint-* if RESUME=auto)")


<a name="Data"></a>
### Dataset

Hub `coder-sft-mix-v1`: conversational `messages` → ChatML `text`. Schema gate + smoke slice before full 136K.


In [ ]:
from datasets import load_dataset
from collections import Counter

REQUIRED_ROLES = ["system", "user", "assistant"]

def valid_messages(msgs) -> bool:
    if not isinstance(msgs, list) or len(msgs) < 3:
        return False
    roles = [m.get("role") for m in msgs[:3]]
    if roles != REQUIRED_ROLES:
        return False
    for m in msgs:
        c = m.get("content")
        if not isinstance(c, str) or not c.strip():
            return False
    return True

print("Loading", DATASET, "…")
try:
    raw = load_dataset(DATASET, token=HF_TOKEN)
except TypeError:
    raw = load_dataset(DATASET)

assert "train" in raw, f"Missing train split; got {list(raw.keys())}"
assert "eval_heldout" in raw, (
    f"Missing eval_heldout; got {list(raw.keys())}"
)
train_raw = raw["train"]
eval_raw = raw["eval_heldout"]

cols = set(train_raw.column_names)
for need in ("messages", "source", "task"):
    assert need in cols, f"train missing column {need}; have {sorted(cols)}"

# Schema gate: first 200 + random 50
n = len(train_raw)
scan_idx = list(range(min(200, n)))
if n > 200:
    rng = random.Random(SEED)
    extra = rng.sample(range(200, n), k=min(50, n - 200))
    scan_idx.extend(extra)
bad = 0
for i in scan_idx:
    if not valid_messages(train_raw[i]["messages"]):
        bad += 1
        if bad <= 3:
            print("Bad row sample idx", i, train_raw[i]["messages"][:1])
assert bad == 0, f"Schema gate failed: {bad}/{len(scan_idx)} rows bad roles/contents"

def keep_row(example):
    return valid_messages(example.get("messages"))

_nproc = 2 if n > 1000 else None
train_ds = train_raw.filter(keep_row, num_proc=_nproc)
eval_ds = eval_raw.filter(keep_row, num_proc=1)

if SMOKE:
    train_ds = train_ds.select(range(min(256, len(train_ds))))
    eval_ds = eval_ds.select(range(min(32, len(eval_ds))))

def to_text(example):
    try:
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    except Exception:
        text = ""
    if not isinstance(text, str):
        text = ""
    return {"text": text}

_map_proc = 2 if len(train_ds) > 64 else None
train_ds = train_ds.map(to_text, num_proc=_map_proc)
eval_ds = eval_ds.map(to_text, num_proc=1)
train_ds = train_ds.filter(lambda ex: bool(ex["text"] and ex["text"].strip()))
eval_ds = eval_ds.filter(lambda ex: bool(ex["text"] and ex["text"].strip()))

task_counts = Counter(train_ds["task"]) if "task" in train_ds.column_names else {}
src_counts = Counter(train_ds["source"]) if "source" in train_ds.column_names else {}
print("train", len(train_ds), "| eval", len(eval_ds), "| SMOKE", SMOKE)
print("task counts:", dict(task_counts))
print("source counts:", dict(src_counts))
sample = train_ds[0]
print("sample task/source:", sample.get("task"), sample.get("source"))
print("text head:", repr(sample["text"][:240]))

train_dataset = train_ds
eval_dataset = eval_ds


<a name="Train"></a>
### Trainer + assistant-only mask + smoke gates

`packing=False` required. `train_on_responses_only` with Qwen markers. Label audit **aborts** before a long train. Timed checkpoints sync to Hugging Face every `CKPT_MINUTES`.


In [ ]:
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

@dataclass
class Phase3State:
    global_step: int = 0
    best_eval_loss: float = float("inf")
    last_eval_loss: float = float("nan")
    tok_approx: int = 0

def write_latest(root: Path, ckpt: Path):
    root.mkdir(parents=True, exist_ok=True)
    (root / "LATEST").write_text(str(ckpt.resolve()))

def read_latest(root: Path):
    f = root / "LATEST"
    if not f.exists():
        return None
    p = Path(f.read_text().strip())
    return p if p.exists() else None

from huggingface_hub import HfApi, snapshot_download

def sync_checkpoint(src: Path, update_latest: bool = True):
    if DRIVE_CKPT is not None:
        dest = DRIVE_CKPT / src.name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(src, dest, dirs_exist_ok=True)
        if update_latest:
            write_latest(DRIVE_CKPT, dest)
        print("Mirrored → Drive:", dest, flush=True)

    if HUB_ADAPTER_ID and HF_TOKEN:
        try:
            # Fix README.md base_model metadata if it contains a local filesystem path
            readme = src / "README.md"
            if readme.is_file():
                txt = readme.read_text()
                if "base_model: /" in txt or "base_model: ." in txt:
                    clean_model_id = MODEL_HUB if "MODEL_HUB" in globals() else "Qwen/Qwen2.5-Coder-7B-Instruct"
                    txt = re.sub(r"base_model:\s*[^]+", f"base_model: {clean_model_id}", txt)
                    readme.write_text(txt)

            api = HfApi(token=HF_TOKEN)
            api.create_repo(repo_id=HUB_ADAPTER_ID, private=True, exist_ok=True)
            print(f"Syncing {src.name} → HF Hub ({HUB_ADAPTER_ID})...", flush=True)
            api.upload_folder(
                folder_path=str(src),
                path_in_repo=src.name,
                repo_id=HUB_ADAPTER_ID,
                repo_type="model",
            )
            print(f"Synced {src.name} → HF Hub ✓", flush=True)
        except Exception as e:
            print(f"Notice: HF Hub sync failed for {src.name}: {e}", flush=True)

def download_latest_hub_checkpoint():
    if not (HUB_ADAPTER_ID and HF_TOKEN):
        return None
    try:
        api = HfApi(token=HF_TOKEN)
        files = api.list_repo_files(repo_id=HUB_ADAPTER_ID, repo_type="model")
    except Exception as e:
        print(f"Notice: could not inspect Hub checkpoints ({e}); trying local resume only.", flush=True)
        return None

    checkpoint_nums = []
    for file in files:
        parts = file.split("/", 1)
        if len(parts) == 2 and parts[0].startswith("checkpoint-") and parts[1] == "trainer_state.json":
            try:
                checkpoint_nums.append((int(parts[0].split("-", 1)[1]), parts[0]))
            except Exception:
                pass
    if not checkpoint_nums:
        return None

    ckpt_name = sorted(checkpoint_nums)[-1][1]
    hub_resume_root = OUT_DIR / "hub_resume"
    hub_resume_root.mkdir(parents=True, exist_ok=True)
    print(f"Downloading Hub resume checkpoint {ckpt_name} from {HUB_ADAPTER_ID}...", flush=True)
    try:
        snapshot_download(
            repo_id=HUB_ADAPTER_ID,
            repo_type="model",
            token=HF_TOKEN,
            allow_patterns=[f"{ckpt_name}/*"],
            local_dir=str(hub_resume_root),
            resume_download=True,
        )
    except Exception as e:
        print(f"Notice: Hub checkpoint download failed ({e}); trying local resume only.", flush=True)
        return None
    ckpt = hub_resume_root / ckpt_name
    return ckpt if is_valid_resume_dir(ckpt) else None

def mirror_to_drive(src: Path, update_latest: bool = True):
    if DRIVE_CKPT is None:
        return
    dest = DRIVE_CKPT / src.name
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(src, dest, dirs_exist_ok=True)
    if update_latest:
        write_latest(DRIVE_CKPT, dest)
    print("Mirrored →", dest, flush=True)

def is_valid_resume_dir(p: Path) -> bool:
    if not p.is_dir():
        return False
    # Never resume from adapters-only final/
    if p.name == "final":
        return False
    return (p / "trainer_state.json").is_file()

def resolve_resume():
    if RESUME in ("none", "", "False", "false"):
        return None
    if RESUME != "auto":
        p = Path(RESUME)
        assert p.exists(), p
        if not is_valid_resume_dir(p):
            raise SystemExit(
                f"RESUME path invalid for trainer resume (need trainer_state.json; "
                f"not final/): {p}"
            )
        return str(p)
    # auto: Drive LATEST → local LATEST → newest checkpoint-* → newest HF checkpoint-*
    candidates = []
    for root in filter(None, [DRIVE_CKPT, OUT_DIR]):
        latest = read_latest(root)
        if latest is not None:
            candidates.append(("latest", latest))
        ckpts = sorted(
            Path(root).glob("checkpoint-*"), key=lambda q: q.stat().st_mtime
        )
        if ckpts:
            candidates.append(("ckpt", ckpts[-1]))
    for kind, c in candidates:
        if is_valid_resume_dir(c):
            print("Resume auto →", c)
            return str(c)
        if (
            c.is_dir()
            and (c / "adapter_config.json").is_file()
            and not (c / "trainer_state.json").is_file()
        ):
            raise SystemExit(
                f"Resume pointer is adapter-only (no trainer_state.json): {c}\\n"
                "Do not use final/ as resume_from_checkpoint. Use checkpoint-* "
                "or clear LATEST and start fresh."
            )
    hub_ckpt = download_latest_hub_checkpoint()
    if hub_ckpt is not None:
        print("Resume auto → Hub", hub_ckpt)
        return str(hub_ckpt)
    print("Resume auto: fresh start")
    return None

class Phase3Callback(TrainerCallback):
    def __init__(self, state: Phase3State):
        self.ps = state
        self.t0 = time.time()
        self.last_t = self.t0
        self.last_save = self.t0
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                torch.cuda.reset_peak_memory_stats(i)

    def _vram(self):
        if not torch.cuda.is_available():
            return 0.0, 0.0
        num_gpus = torch.cuda.device_count()
        alloc = sum(torch.cuda.memory_allocated(i) for i in range(num_gpus)) / 1e9
        peak = sum(torch.cuda.max_memory_allocated(i) for i in range(num_gpus)) / 1e9
        return alloc, peak

    def on_step_end(self, args, state, control, **kwargs):
        self.ps.global_step = state.global_step
        world = max(int(os.environ.get("WORLD_SIZE", torch.cuda.device_count() if torch.cuda.is_available() else 1)), 1)
        self.ps.tok_approx += (
            args.per_device_train_batch_size
            * args.gradient_accumulation_steps
            * MAX_SEQ_LEN
            * world
        )
        now = time.time()
        if (now - self.last_t) >= 30 or state.global_step % max(args.logging_steps, 1) == 0:
            alloc, peak = self._vram()
            loss = "n/a"
            if state.log_history:
                v = state.log_history[-1].get("loss")
                if isinstance(v, (int, float)):
                    loss = f"{v:.4f}"
            print(
                f"step={state.global_step} loss={loss} "
                f"VRAM={alloc:.1f}G peak={peak:.1f}G "
                f"elapsed={(now - self.t0) / 60:.1f}m",
                flush=True,
            )
            self.last_t = now
        if (now - self.last_save) >= CKPT_MINUTES * 60:
            control.should_save = True
            self.last_save = now
            print(f"Timed checkpoint ({CKPT_MINUTES:.0f} min)", flush=True)
        if WALL_CLOCK_HOURS and WALL_CLOCK_HOURS > 0:
            elapsed_h = (now - self.t0) / 3600.0
            if elapsed_h >= WALL_CLOCK_HOURS:
                print(
                    f"Wall-clock limit {WALL_CLOCK_HOURS:.1f}h reached "
                    f"(elapsed {elapsed_h:.2f}h); saving and stopping before Colab kill.",
                    flush=True,
                )
                control.should_training_stop = True
                control.should_save = True
        return control

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        metrics = metrics or {}
        v = metrics.get("eval_loss")
        if isinstance(v, (int, float)):
            self.ps.last_eval_loss = float(v)
            if math.isnan(v) or math.isinf(v):
                print("WARNING: eval_loss is nan/inf — check masking / data.", flush=True)
            else:
                print(f"eval_loss={v:.4f} (step {state.global_step})", flush=True)
                if v < self.ps.best_eval_loss:
                    self.ps.best_eval_loss = float(v)
        return control

    def on_save(self, args, state, control, **kwargs):
        ckpt = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if ckpt.exists():
            (ckpt / "phase3_state.json").write_text(
                json.dumps(asdict(self.ps), indent=2)
            )
            write_latest(OUT_DIR, ckpt)
            sync_checkpoint(ckpt, update_latest=True)
        return control

phase_state = Phase3State()
resume_path = resolve_resume()
if resume_path:
    sf = Path(resume_path) / "phase3_state.json"
    if not sf.exists():
        sf = Path(resume_path) / "phase2_state.json"
    if sf.exists():
        d = json.loads(sf.read_text())
        phase_state = Phase3State(
            **{k: d[k] for k in Phase3State.__dataclass_fields__ if k in d}
        )
        print("Restored phase3_state step=", phase_state.global_step)

# Optimizer: QLoRA-friendly (paged 8-bit Adam when bitsandbytes is available)
optim = "adamw_torch"
try:
    import bitsandbytes  # noqa: F401
    optim = "paged_adamw_8bit"
except Exception:
    try:
        from transformers.training_args import OptimizerNames
        names = {n.value if hasattr(n, "value") else str(n) for n in OptimizerNames}
        if "adamw_8bit" in names or "adamw_8bit" in str(names):
            optim = "adamw_8bit"
    except Exception:
        pass
print("optim:", optim)

# Precision & multi-GPU options
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

sft_kwargs = dict(
    output_dir=str(OUT_DIR),
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    bf16=use_bf16,
    fp16=use_fp16,
    optim=optim,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
    dataset_num_proc=2,
    packing=PACKING,
    padding_free=False,
    dataset_text_field="text",
    max_seq_length=None if not PACKING else MAX_SEQ_LEN,
    max_length=None if not PACKING else MAX_SEQ_LEN,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
        load_best_model_at_end=False,
    do_eval=True,
    loss_type="nll",
    use_liger_kernel=False,
)
if SMOKE or MAX_STEPS is not None:
    sft_kwargs["max_steps"] = int(MAX_STEPS if MAX_STEPS is not None else 30)
    sft_kwargs["warmup_steps"] = 5
else:
    sft_kwargs["num_train_epochs"] = NUM_TRAIN_EPOCHS
    sft_kwargs["warmup_ratio"] = 0.03
    sft_kwargs["max_steps"] = -1

if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    sft_kwargs["ddp_find_unused_parameters"] = False

if HUB_ADAPTER_ID and HF_TOKEN:
    sft_kwargs.update(
        push_to_hub=True,
        hub_model_id=HUB_ADAPTER_ID,
        hub_strategy="every_save",
        hub_private_repo=True,
    )

def build_sft_config(kwargs):
    kwargs = dict(kwargs)
    if not kwargs.get("packing", False):
        kwargs["max_seq_length"] = None
        kwargs["max_length"] = None
        kwargs["padding_free"] = False
        kwargs["loss_type"] = "nll"
        kwargs["use_liger_kernel"] = False
    try:
        return SFTConfig(**kwargs)
    except TypeError:
        pass
    if "eval_strategy" in kwargs:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
        try:
            return SFTConfig(**kwargs)
        except TypeError:
            pass
    if "max_seq_length" in kwargs:
        kwargs["max_length"] = kwargs.pop("max_seq_length")
        try:
            return SFTConfig(**kwargs)
        except TypeError:
            pass
    for k in (
        "packing", "padding_free", "dataset_text_field", "logging_first_step",
        "max_length", "max_seq_length", "do_eval", "hub_private_repo",
        "loss_type", "use_liger_kernel",
    ):
        kwargs.pop(k, None)
    return SFTConfig(**kwargs)

args = build_sft_config(sft_kwargs)
if not PACKING:
    for attr, value in (("packing", False), ("padding_free", False), ("max_length", None), ("max_seq_length", None), ("loss_type", "nll"), ("use_liger_kernel", False)):
        if hasattr(args, attr):
            setattr(args, attr, value)
    print(
        "SFT config normalized:",
        "packing=", getattr(args, "packing", None),
        "padding_free=", getattr(args, "padding_free", None),
        "max_length=", getattr(args, "max_length", None),
        "max_seq_length=", getattr(args, "max_seq_length", None),
        "loss_type=", getattr(args, "loss_type", None),
        "use_liger_kernel=", getattr(args, "use_liger_kernel", None),
    )

def patch_trl_chunked_ce_for_unsloth():
    """Avoid TRL chunked-CE crashing when Unsloth exposes forward as functools.partial."""
    import functools
    try:
        import trl.trainer.sft_trainer as trl_sft_trainer
    except Exception as e:
        print(f"Notice: could not import TRL sft_trainer for chunked CE patch: {e}", flush=True)
        return

    original = getattr(trl_sft_trainer, "_patch_chunked_ce_lm_head", None)
    if original is None:
        return
    if getattr(original, "_phase3_unsloth_safe", False):
        safe_patch = original
    else:
        def safe_patch(target, *patch_args, **patch_kwargs):
            forward = getattr(target, "forward", None)
            if isinstance(forward, functools.partial) or not hasattr(forward, "__func__"):
                print(
                    "Skipping TRL chunked CE lm_head patch for Unsloth partial forward; "
                    "using standard model forward.",
                    flush=True,
                )
                return target
            return original(target, *patch_args, **patch_kwargs)
        safe_patch._phase3_unsloth_safe = True
        trl_sft_trainer._patch_chunked_ce_lm_head = safe_patch

    # Unsloth may call a compiled SFTTrainer class whose globals captured the old function.
    patched = 0
    for cls in getattr(SFTTrainer, "__mro__", (SFTTrainer,)):
        init = getattr(cls, "__init__", None)
        globs = getattr(init, "__globals__", None)
        if isinstance(globs, dict) and "_patch_chunked_ce_lm_head" in globs:
            globs["_patch_chunked_ce_lm_head"] = safe_patch
            patched += 1
    print(f"TRL chunked CE compatibility patch active (patched scopes={patched})", flush=True)

patch_trl_chunked_ce_for_unsloth()
trainer_kwargs = dict(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    callbacks=[Phase3Callback(phase_state)],
)
try:
    trainer = SFTTrainer(**trainer_kwargs)
except TypeError:
    trainer_kwargs.pop("processing_class", None)
    trainer_kwargs["tokenizer"] = tokenizer
    trainer = SFTTrainer(**trainer_kwargs)

# Assistant-only loss (Qwen ChatML markers — include trailing newline)
INSTRUCTION_PART = "<|im_start|>user\n"
RESPONSE_PART = "<|im_start|>assistant\n"
trainer = train_on_responses_only(
    trainer,
    instruction_part=INSTRUCTION_PART,
    response_part=RESPONSE_PART,
)

# ── Hard gates (abort before long train) ───────────────────────────────────
packing_on = bool(getattr(trainer.args, "packing", False))
assert (not packing_on) and (PACKING is False), (
    f"FATAL: packing must be False for Phase 3 CoT; trainer.args.packing={packing_on}"
)
print("packing OFF (required)")

# Label audit via train dataloader (post train_on_responses_only)
dl = trainer.get_train_dataloader()
batch = next(iter(dl))
assert "labels" in batch, "Train batch missing labels — cannot audit response mask"
labels_t = batch["labels"]
input_ids_t = batch["input_ids"]
n_rows = min(8, int(labels_t.shape[0]))
fracs = []
decoded_example = None
think_open_count = 0
think_closed_count = 0
for i in range(n_rows):
    labs = labels_t[i].tolist()
    ids = input_ids_t[i].tolist()
    total = len(labs)
    trainable = sum(1 for x in labs if x != -100)
    frac = trainable / max(total, 1)
    fracs.append(frac)
    pieces = [
        tokenizer.decode([tid]) if lab != -100 else " "
        for tid, lab in zip(ids, labs)
    ]
    decoded_row = "".join(pieces)
    if "<think>" in decoded_row:
        think_open_count += 1
    if "</think>" in decoded_row:
        think_closed_count += 1
    if decoded_example is None:
        decoded_example = decoded_row

assert fracs, "Label audit produced no rows"
for frac in fracs:
    if frac <= 0.0:
        # Print token lengths to help debug truncation
        lengths = [len(tokenizer.encode(train_dataset[j]["text"])) for j in range(min(4, len(train_dataset)))]
        raise SystemExit(
            "FATAL: trainable labels fraction is 0 (all -100). "
            "Check train_on_responses_only markers "
            f"{INSTRUCTION_PART!r} / {RESPONSE_PART!r} and ChatML formatting. "
            f"Sample token lengths: {lengths}"
        )
    if not (0.001 < frac < 0.995):
        lengths = [len(tokenizer.encode(train_dataset[j]["text"])) for j in range(min(4, len(train_dataset)))]
        raise SystemExit(
            f"FATAL: trainable label fraction {frac:.3f} outside sane CoT band (0.1%–99.5%). "
            f"Likely masking failure or an almost prompt-free row. Sample token lengths: {lengths}"
        )

print(
    f"Label audit OK over {len(fracs)} rows; trainable frac min/max="
    f"{min(fracs):.3f}/{max(fracs):.3f}"
)
print("Masked decode (spaces = ignored labels) head:")
print(repr((decoded_example or "")[:500]))
assert decoded_example and any(ch.strip() for ch in decoded_example), (
    "Masked decode empty — assistant span not visible"
)
assert think_open_count > 0, (
    "Masked decode does not include <think>; check response marker and CoT formatting"
)
if think_closed_count < think_open_count:
    print(
        f"WARNING: {think_open_count - think_closed_count}/{think_open_count} audited CoT rows "
        "did not include </think> in the masked batch view. This can happen when a long row "
        "is truncated after the reasoning start; source rows were validated before tokenization.",
        flush=True,
    )

print("resume_path=", resume_path)
print(
    "eval_steps", EVAL_STEPS, "save_steps", SAVE_STEPS,
    "CKPT_MINUTES", CKPT_MINUTES, "effective_batch", BATCH * ACCUM,
)


### Train


In [ ]:
# @title Show current memory stats + train
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if num_gpus > 1:
    total_max_mem, total_start_mem = 0.0, 0.0
    for i in range(num_gpus):
        props = torch.cuda.get_device_properties(i)
        start_mem = round(torch.cuda.max_memory_reserved(i) / 1024 / 1024 / 1024, 3)
        max_mem = round(props.total_memory / 1024 / 1024 / 1024, 3)
        total_max_mem += max_mem
        total_start_mem += start_mem
        print(f"GPU {i} = {props.name}. Max memory = {max_mem} GB. Reserved = {start_mem} GB.")
    print(f"Total GPUs = {num_gpus}. Total Max memory = {round(total_max_mem, 3)} GB. Reserved = {round(total_start_mem, 3)} GB.")
    start_gpu_memory = total_start_mem
    max_memory = total_max_mem
elif num_gpus == 1:
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
    print(f"{start_gpu_memory} GB of memory reserved.")
else:
    start_gpu_memory = 0.0
    max_memory = 1.0
    print("Using CPU")

try:
    trainer_stats = trainer.train(resume_from_checkpoint=resume_path)
except torch.cuda.OutOfMemoryError as e:
    raise SystemExit(
        "CUDA OOM during Phase 3 CoT. Cut BATCH (e.g. 4→2) or raise ACCUM to keep "
        f"effective batch ≈{BATCH * ACCUM}. Original error: {e}"
    ) from e


In [ ]:
# @title Show final memory and time stats
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if num_gpus > 0:
    used_memory = round(sum(torch.cuda.max_memory_reserved(i) for i in range(num_gpus)) / 1024 / 1024 / 1024, 3)
else:
    used_memory = 0.0
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max(max_memory, 0.001) * 100, 3)
lora_percentage = round(used_memory_for_lora / max(max_memory, 0.001) * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")
print("global_step≈", phase_state.global_step, "tok_approx≈", f"{phase_state.tok_approx:,}")
print("best_eval_loss", phase_state.best_eval_loss, "last_eval_loss", phase_state.last_eval_loss)


<a name="Inference"></a>
### Inference smoke (4 tasks)

Format/sanity only — not a quality claim. Always `add_generation_prompt=True`.


In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM_PROMPTS = {
    "code_reasoning": "You are an expert coding assistant. Think through problems step-by-step in <think> tags before providing your solution.",
    "security_reasoning": "You are a security analyst. Analyze vulnerabilities step-by-step in <think> tags before providing your findings and remediation.",
    "code_review": "You are a senior GitHub code reviewer. Think through the code step-by-step in <think> tags before writing your review.",
    "code_review_fix": "You apply GitHub review feedback. Think through the fix step-by-step in <think> tags before outputting the corrected code.",
}

TEST_TASKS = [
    (
        "code_reasoning",
        "Implement a Python function `find_longest_substring(s: str) -> int` that finds the length of the longest substring without repeating characters."
    ),
    (
        "security_reasoning",
        "Analyze the security implications of: `eval(user_input)` in a Python web backend."
    ),
    (
        "code_review",
        "Language: python\nFile: auth.py\n```diff\n@@ -10,3 +10,3 @@\ndef check_password(stored_hash, user_password):\n-    return stored_hash == hashlib.md5(user_password.encode()).hexdigest()\n+    return bcrypt.checkpw(user_password.encode(), stored_hash.encode())\n```\nReview this pull request change."
    ),
    (
        "code_review_fix",
        "Language: python\nOriginal code:\n```python\n@app.route('/user')\ndef get_user():\n    user_id = request.args.get('id')\n    return db.execute(f'SELECT * FROM users WHERE id={user_id}')\n```\nReviewer feedback: SQL injection risk. Use parameterized query.\nApply feedback and fix the code."
    ),
]

for task, prompt in TEST_TASKS:
    print("=" * 60)
    print(f"TASK: {task}")
    print("=" * 60)
    sys_p = SYSTEM_PROMPTS.get(task, "Think step-by-step in <think> tags before providing your answer.")
    messages = [
        {"role": "system", "content": sys_p},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        use_cache=True,
        temperature=0.3,
    )
    res = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(res)
    print("\n")


<a name="Save"></a>
### Save adapters (every session end)

Adapters only (`final/`) — not a full merge. `LATEST` stays on newest `checkpoint-*` for auto-resume. Run Cell 22 only after training is fully done.


In [ ]:
final_dir = OUT_DIR / "final"
final_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

# Sync adapters, but do NOT point LATEST at final/ (resume needs checkpoint-*)
sync_checkpoint(final_dir, update_latest=False)

# Ensure LATEST still prefers newest checkpoint-* if present
for root in filter(None, [OUT_DIR, DRIVE_CKPT]):
    ckpts = sorted(Path(root).glob("checkpoint-*"), key=lambda p: p.stat().st_mtime)
    if ckpts:
        write_latest(root, ckpts[-1])
        print("LATEST →", ckpts[-1])
    else:
        print("No checkpoint-* under", root, "— LATEST unchanged (do not use final/ for resume)")

meta = {
    "phase": "phase3_cot_qlora",
    "base_model_dir": str(MODEL_DIR),
    "base_hub": MODEL_HUB,
    "dataset": DATASET,
    "hub_adapter_id": HUB_ADAPTER_ID,
    "packing": False,
    "load_in_4bit": LOAD_IN_4BIT,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lr": LR,
    "max_seq_len": MAX_SEQ_LEN,
    "batch": BATCH,
    "accum": ACCUM,
    "smoke": SMOKE,
    "global_step": phase_state.global_step,
    "best_eval_loss": phase_state.best_eval_loss,
    "resume": RESUME,
    "ckpt_minutes": CKPT_MINUTES,
}
(final_dir / "phase3_meta.json").write_text(json.dumps(meta, indent=2))
(final_dir / "phase3_state.json").write_text(json.dumps(asdict(phase_state), indent=2))
print("Saved adapters →", final_dir)

if HUB_ADAPTER_ID and HF_TOKEN:
    model.push_to_hub(HUB_ADAPTER_ID, private=True, token=HF_TOKEN)
    tokenizer.push_to_hub(HUB_ADAPTER_ID, private=True, token=HF_TOKEN)
    print("Pushed adapters →", HUB_ADAPTER_ID)

print(
    "Reminder: multi-session resume uses checkpoint-* folders on Hugging Face via RESUME=auto. "
    "Set RUN_FINAL_MERGE=True and run Cell 22 only after training is fully done."
)


### Next session

1. Re-run **Install** + **Config** with `SMOKE = False`, `RESUME = "auto"`, same `OUT_DIR` / Drive paths.
2. **1h Hugging Face checkpoint syncs** (`CKPT_MINUTES=60`) are the continuity mechanism across Kaggle/Colab reconnects.
3. After smoke: raise `BATCH` toward 4–8 using peak VRAM; keep effective batch ~32–64 via `ACCUM`.
4. If loss flat / eval rises: lower LR or raise LoRA `r`; if all-`-100` after an Unsloth bump: re-check response markers + `\n`.
5. When the full epoch (or target) is finished: set `RUN_FINAL_MERGE = True` and run **Cell 22 once**.

See `fine-tune/FINE_TUNE_DECISIONS.md` §6–7.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 22 — Optional final merge (LAST RUN ONLY)
# Merge Phase-2 SFT LoRA into the CPT-merged BF16 domain base → standalone
# BF16 instruct-capable weights. Skip until training is complete.
# Does not affect resume. Gated by RUN_FINAL_MERGE (default False).
# Pattern: fine-tune/merge_cpt_lora_colab.py (BF16 base, not 4-bit).
# ═══════════════════════════════════════════════════════════════════════════

if not RUN_FINAL_MERGE:
    print(
        "RUN_FINAL_MERGE=False — skipping Cell 22 final merge. "
        "Flip to True only after the last training session."
    )
else:
    from peft import PeftModel
    from unsloth.chat_templates import get_chat_template as _get_chat_template

    def _require_adapter_dir(adapter_dir: Path) -> None:
        if not adapter_dir.is_dir():
            raise SystemExit(f"Adapter dir missing: {adapter_dir}")
        cfg = adapter_dir / "adapter_config.json"
        weights = list(adapter_dir.glob("adapter_model*.safetensors")) + list(
            adapter_dir.glob("adapter_model.bin")
        )
        if not cfg.is_file():
            raise SystemExit(f"Missing adapter_config.json in {adapter_dir}")
        if not weights:
            raise SystemExit(
                f"Missing adapter weights in {adapter_dir} "
                "(expected adapter_model.safetensors or .bin)"
            )
        if adapter_dir.name.startswith("checkpoint-"):
            raise SystemExit(
                f"Refusing mid-train checkpoint as merge adapter: {adapter_dir}\n"
                "Prefer OUT_DIR/final (or Drive …/final). "
                "Override only by copying that checkpoint into a final/ path."
            )

    # Resolve adapter: Drive final → local final
    adapter_candidates = []
    if DRIVE_CKPT is not None:
        adapter_candidates.append(DRIVE_CKPT / "final")
    adapter_candidates.append(OUT_DIR / "final")
    adapter_dir = None
    for c in adapter_candidates:
        if (c / "adapter_config.json").is_file():
            adapter_dir = c
            break
    if adapter_dir is None:
        raise SystemExit(
            "No SFT adapter final/ found under Drive or OUT_DIR. "
            "Run Cell 20 save before Cell 22 merge."
        )
    _require_adapter_dir(adapter_dir)

    # Resolve CPT-merged BF16 base (same preference as Cell 07)
    if USE_DRIVE and local_model_complete(MODEL_DRIVE):
        merge_base = MODEL_DRIVE
    elif local_model_complete(MODEL_CACHE):
        merge_base = MODEL_CACHE
    elif local_model_complete(MODEL_DIR):
        merge_base = MODEL_DIR
    else:
        raise SystemExit(
            "CPT-merged base not found for BF16 merge. "
            "Restore Drive merge or Hub snapshot first."
        )

    out_dir = Path(MERGE_OUT_DIR)
    if out_dir.exists():
        if not FORCE_MERGE:
            raise SystemExit(
                f"Merge out_dir already exists: {out_dir}\n"
                "Set FORCE_MERGE=True to delete and rewrite, or change MERGE_OUT_DIR."
            )
        print(f"FORCE_MERGE: removing {out_dir}", flush=True)
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Free 4-bit train model before BF16 load (A100 40GB+)
    print("Freeing train model from GPU…", flush=True)
    try:
        del model
    except NameError:
        pass
    try:
        del trainer
    except NameError:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Base (BF16):", merge_base, flush=True)
    print("Adapter:    ", adapter_dir, flush=True)
    print("Out:        ", out_dir, flush=True)

    print("Loading CPT-merged base BF16 (load_in_4bit=False)…", flush=True)
    merge_model, merge_tok = FastLanguageModel.from_pretrained(
        model_name=str(merge_base),
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=False,
        token=HF_TOKEN,
    )
    merge_tok = _get_chat_template(merge_tok, chat_template="qwen-2.5")

    print("Attaching Phase-2 SFT LoRA…", flush=True)
    merge_model = PeftModel.from_pretrained(merge_model, str(adapter_dir))
    print("Merging + unloading…", flush=True)
    merge_model = merge_model.merge_and_unload()

    print("Saving merged BF16 weights…", flush=True)
    saved = False
    if hasattr(merge_model, "save_pretrained_merged"):
        try:
            merge_model.save_pretrained_merged(
                str(out_dir),
                merge_tok,
                save_method="merged_16bit",
            )
            saved = True
            print("Saved via Unsloth save_pretrained_merged(merged_16bit).", flush=True)
        except Exception as e:
            print(f"Unsloth merged save failed ({e}); falling back to HF save.", flush=True)

    if not saved:
        merge_model.save_pretrained(str(out_dir), safe_serialization=True)
        merge_tok.save_pretrained(str(out_dir))
        print("Saved via model.save_pretrained + tokenizer.", flush=True)

    # Ensure ChatML tokenizer is on disk for inference-ready artifact
    merge_tok.save_pretrained(str(out_dir))

    merge_meta = {
        "phase": "sft_merged_instruct_capable",
        "cpt_base": str(merge_base),
        "sft_adapter": str(adapter_dir),
        "out_dir": str(out_dir),
        "hub_merged_sft_id": HUB_MERGED_SFT_ID or None,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "dataset": DATASET,
        "notes": "Merged SFT adapters into CPT-merged BF16 base; ChatML tokenizer saved.",
    }
    (out_dir / "merge_meta.json").write_text(json.dumps(merge_meta, indent=2))

    has_cfg = (out_dir / "config.json").is_file()
    weight_files = list(out_dir.glob("*.safetensors")) + list(
        out_dir.glob("pytorch_model*.bin")
    )
    large = [p for p in weight_files if p.stat().st_size > 50_000_000]
    if not has_cfg or not large:
        raise SystemExit(
            f"Merge output looks incomplete under {out_dir} "
            f"(config.json={has_cfg}, large_weight_files={len(large)})."
        )
    gb = sum(p.stat().st_size for p in out_dir.rglob("*") if p.is_file()) / 1e9
    print(f"Merge complete ({gb:.1f} GB) → {out_dir}", flush=True)

    if HUB_MERGED_SFT_ID:
        if not HF_TOKEN:
            raise SystemExit("HUB_MERGED_SFT_ID set but no HF token")
        print(f"Pushing merged model → {HUB_MERGED_SFT_ID} (private)…", flush=True)
        merge_model.push_to_hub(HUB_MERGED_SFT_ID, private=True, token=HF_TOKEN)
        merge_tok.push_to_hub(HUB_MERGED_SFT_ID, private=True, token=HF_TOKEN)
        print("Hub push done.", flush=True)

    print("Phase 3 final merge done. Artifact is inference-ready (ChatML).", flush=True)
